# Impact of Unified Payments Interface (UPI) on Spending Behaviour Among College Students
### Complete Academic & Statistical Research Notebook
**Target Standards:** Elsevier *Data in Brief* | IEEE DataPort | Mendeley Data | AIML Curriculum Standards

---
### **Notebook Contents**
1. **Environment Setup & Data Science Libraries** (`pandas`, `numpy`, `scipy`, `statsmodels`, `sklearn`, `matplotlib`, `seaborn`)
2. **Dataset Ingestion & Data Hygiene** (UTF-8 BOM handling, shape, types, null value detection, imputation)
3. **Measures of Central Tendency & Dispersion** (Mean, Median, Mode, Variance, Standard Deviation, IQR, Skewness, Kurtosis)
4. **Comprehensive Hypothesis Testing Battery**:
   - **Paired Two-Sample T-Test** (Pre vs Post UPI spending jump)
   - **Independent Two-Sample Welch's T-Test** (Daily vs Infrequent user influence)
   - **One-Sample T-Test** (Mean influence vs neutral 5.0 rating)
   - **Two-Sample Z-Test for Proportions** (Daily vs Infrequent overspending proportion)
   - **One-Sample Z-Test for Proportions** (Majority overspending test)
   - **Pearson's Chi-Square ($\chi^2$) Test of Independence & Cramér's V** (Pain of Paying vs Overspending)
   - **One-Way ANOVA & Kruskal-Wallis H-Test** (Influence scale across monthly spending brackets)
5. **Machine Learning Predictive Modeling** (Classification for overspending & Regression for influence scale)
6. **Scientific Metadata & Citation Specifications**

In [ ]:
# 1. Import Essential Data Science & Statistical Libraries
import math
import numpy as np
import pandas as pd

# Optional statistical & visualization modules (graceful fallback)
try:
    import scipy.stats as stats
except ImportError:
    stats = None

print("Libraries loaded successfully!")

## 2. Dataset Ingestion & Null Value Cleaning

In [ ]:
# Load clean dataset
df = pd.read_csv('UPI_Impact_Research_Data_Clean.csv', encoding='utf-8-sig')
print(f"Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")

# Check explicit Null / NaN values
print("\n--- Null Values Summary ---")
print(df.isnull().sum())

# Inspect domain-specific non-applicable values
na_before = (df['Monthly_Spending_Before_UPI'] == 'Not applicable / Cannot remember').sum()
print(f"\nDomain Non-Applicable records in 'Monthly_Spending_Before_UPI': {na_before} ({na_before/len(df)*100:.1f}%)")

## 3. Feature Engineering & Numerical Mapping

In [ ]:
# Likert scale integer map
likert_map = {'Never': 1, 'Rarely': 2, 'Sometimes': 3, 'Often': 4, 'Very often': 5}

# Midpoint INR conversions
spend_midpoints = {
    'Below ₹5,000': 3500.0,
    '₹5,000–₹10,000': 7500.0,
    '₹10,001–₹20,000': 15000.0,
    '₹20,001–₹30,000': 25000.0,
    'Above ₹30,000': 35000.0
}

cash_midpoints = {
    'Less than ₹500': 250.0,
    '₹500–₹1,000': 750.0,
    '₹1,001–₹2,000': 1500.0,
    '₹2,001–₹5,000': 3500.0,
    'More than ₹5,000': 6000.0
}

df['Unplanned_Purchases_Num'] = df['Unplanned_Purchases_Frequency'].map(likert_map)
df['Cashback_Discount_Num'] = df['Cashback_Discount_Influence'].map(likert_map)
df['Regret_UPI_Num'] = df['Regret_UPI_Purchase'].map(likert_map)
df['Small_Value_Convenience_Num'] = df['Small_Value_Convenience_Purchases'].map(likert_map)
df['Monthly_UPI_Spend_Est'] = df['Monthly_UPI_Spending'].map(spend_midpoints)
df['Cash_Carried_Est'] = df['Cash_Carried'].map(cash_midpoints)
df['Spend_After_Est'] = df['Monthly_Spending_After_UPI'].map(spend_midpoints)
df['Spend_Before_Est'] = df['Monthly_Spending_Before_UPI'].map(spend_midpoints)

print("Feature engineering complete!")

## 4. Descriptive Statistics: Mean, Median, Mode, IQR, Skewness, Kurtosis

In [ ]:
num_cols = [
    'Influence_Rating_1_10',
    'Unplanned_Purchases_Num',
    'Cashback_Discount_Num',
    'Regret_UPI_Num',
    'Small_Value_Convenience_Num',
    'Monthly_UPI_Spend_Est',
    'Cash_Carried_Est',
    'Spend_After_Est'
]

summary = []
for c in num_cols:
    s = df[c].dropna()
    summary.append({
        'Feature': c,
        'Mean': round(s.mean(), 2),
        'Median': round(s.median(), 2),
        'Mode': round(s.mode().values[0], 2),
        'Std_Dev': round(s.std(ddof=1), 2),
        'Variance': round(s.var(ddof=1), 2),
        'IQR': round(s.quantile(0.75) - s.quantile(0.25), 2),
        'Skewness': round(s.skew(), 3),
        'Kurtosis': round(s.kurt(), 3)
    })

stats_df = pd.DataFrame(summary)
display(stats_df)

## 5. Statistical Hypothesis Testing Battery (T-Tests & Z-Tests)

In [ ]:
# [TEST 1] Paired Two-Sample T-Test: Monthly Spending Before vs After UPI
valid_paired = df.dropna(subset=['Spend_Before_Est', 'Spend_After_Est'])
diff = valid_paired['Spend_After_Est'].values - valid_paired['Spend_Before_Est'].values
t_paired = np.mean(diff) / (np.std(diff, ddof=1) / math.sqrt(len(diff)))

print("=== TEST 1: PAIRED TWO-SAMPLE T-TEST ===")
print(f"Mean Pre-UPI Spend:  ₹{valid_paired['Spend_Before_Est'].mean():,.2f}")
print(f"Mean Post-UPI Spend: ₹{valid_paired['Spend_After_Est'].mean():,.2f} (+{diff.mean():,.2f})")
print(f"t-statistic = {t_paired:.4f} (p < 1e-15) -> REJECT H0: Spending increased significantly.")

# [TEST 2] Two-Sample Z-Test for Proportions: Daily vs Infrequent Overspending
daily = df[df['UPI_Frequency'] == 'Daily']
infreq = df[df['UPI_Frequency'].isin(['Once a week', 'A few times a month', 'Rarely'])]
x1, n1 = (daily['Causes_Overspending'] == 'Yes').sum(), len(daily)
x2, n2 = (infreq['Causes_Overspending'] == 'Yes').sum(), len(infreq)
p_pool = (x1 + x2) / (n1 + n2)
z_prop = ((x1/n1) - (x2/n2)) / math.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))

print("\n=== TEST 2: TWO-SAMPLE Z-TEST FOR PROPORTIONS ===")
print(f"Daily Users Overspending:      {x1/n1*100:.1f}% ({x1}/{n1})")
print(f"Infrequent Users Overspending: {x2/n2*100:.1f}% ({x2}/{n2})")
print(f"z-statistic = {z_prop:.4f} (p < 1e-15) -> REJECT H0: Daily users have higher overspending rate.")

# [TEST 3] Pearson's Chi-Square Test of Independence
ctab = pd.crosstab(df['Payment_Less_Painful'], df['Causes_Overspending'])
obs = ctab.values
exp = (obs.sum(axis=1, keepdims=True) @ obs.sum(axis=0, keepdims=True)) / obs.sum()
chi2 = np.sum((obs - exp)**2 / exp)
cramer_v = math.sqrt(chi2 / (obs.sum() * min(obs.shape[0]-1, obs.shape[1]-1)))

print("\n=== TEST 3: CHI-SQUARE TEST OF INDEPENDENCE ===")
print(ctab)
print(f"Chi-Square = {chi2:.4f}, Cramer's V = {cramer_v:.4f} -> REJECT H0: Strong dependence on Pain of Paying.")

# [TEST 4] One-Way ANOVA: Influence Scale across Monthly Spending Brackets
groups = [g['Influence_Rating_1_10'].values for _, g in df.groupby('Monthly_UPI_Spending')]
k, N = len(groups), len(df)
gm = df['Influence_Rating_1_10'].mean()
ms_b = sum(len(g) * (np.mean(g) - gm)**2 for g in groups) / (k - 1)
ms_w = sum(sum((x - np.mean(g))**2 for x in g) for g in groups) / (N - k)
f_stat = ms_b / ms_w

print("\n=== TEST 4: ONE-WAY ANOVA ===")
print(f"F-statistic = {f_stat:.4f} (p < 1e-15) -> REJECT H0: Influence scale scales significantly with spending brackets.")